# S3 J5 - Partage d'état entre agents

Objectifs

Comprendre :

- Agent State
- Shared State
- Session State
- Redis
- PostgreSQL
- Mémoire court terme
- Mémoire long terme
- Coordination multi-agent
```
- Comment plusieurs agents partagent un état ?
- Comment éviter les incohérences ?
- Comment gérer la mémoire ?
- Comment éviter que chaque agent refasse le travail ?
```

# Semaine 3 - Jour 5

# Sharing State Between Agents

Objectif :

Comprendre comment plusieurs agents
partagent des informations.


# Pourquoi ce sujet est important ?

Agent unique :
```
Utilisateur
↓
Agent
↓
Réponse

Simple.

Multi-agent :

Utilisateur
↓
Supervisor
↓
Planner
↓
Researcher
↓
Writer
```
Qui garde l'état ?

# Définition

State = ensemble des informations nécessaires
à la poursuite d'une tâche.

# Exemple

Utilisateur :

"Prépare un voyage à Rome."

Le Planner produit :

- Vol
- Hôtel
- Activités

Le Researcher doit connaître ce plan.



In [1]:
task = {

    "destination": "Rome",

    "steps": [
        "flight",
        "hotel",
        "activities"
    ]
}

# Solution naïve

Chaque agent possède sa copie.

Mauvaise idée.

In [2]:
planner_state = task.copy()
researcher_state = task.copy()
writer_state = task.copy()

# Pourquoi c'est mauvais ?

Les copies divergent rapidement.

In [3]:
planner_state["budget"] = 1200

print(researcher_state)

{'destination': 'Rome', 'steps': ['flight', 'hotel', 'activities']}


# Shared State

Tous les agents lisent
une source unique.

In [4]:
shared_state = {

    "destination": "Rome",

    "budget": None,

    "hotel": None
}

class Planner:

    def run(self):

        shared_state["budget"] = 1200

class Researcher:

    def run(self):

        return shared_state["budget"]

Planner().run()

Researcher().run()



1200

# Architecture classique
```
           Redis

             ▲

   ┌─────────┼─────────┐

   ▼         ▼         ▼

Planner  Researcher  Writer
```

# Pourquoi Redis ?

- très rapide
- partagé
- simple
- TTL

In [5]:
import redis

redis_client = redis.Redis(
    host="localhost",
    port=6379
)

redis_client.set(
    "session:123:budget",
    "1200"
)

redis_client.get(
    "session:123:budget"
)

ConnectionError: Error 61 connecting to localhost:6379. Connection refused.

# Structure recommandée

In [ ]:
session_state = {

    "user_id": "123",

    "conversation_id": "abc",

    "current_task": "trip",

    "budget": 1200
}

# Session State

Durée :

minutes ou heures.

# Long Term Memory

Durée :

jours
mois
années

# Exemple

Préférence utilisateur :

"Je préfère Air France."

Doit être sauvegardée.

```
CREATE TABLE memories (

    id BIGSERIAL PRIMARY KEY,

    user_id TEXT,

    content TEXT,

    created_at TIMESTAMP
);
```

# Redis vs PostgreSQL

Redis

→ état courant

PostgreSQL

→ mémoire durable

# Pattern Supervisor

In [ ]:
class Supervisor:

    def __init__(self):

        self.state = shared_state

    def run(self):

        Planner().run()

        Researcher().run()

# Que faut-il stocker ?

In [ ]:
important_keywords = [

    "objectif",

    "préférence",

    "projet",

    "contrainte"
]

def should_store(text):

    return any(
        word in text.lower()
        for word in important_keywords
    )

# Conflits d'écriture

Deux agents peuvent modifier
la même donnée.

In [6]:
shared_state["budget"] = 1200

shared_state["budget"] = 1500

# Solutions

- verrou Redis
- versioning
- event sourcing

# Architecture Production

```
Utilisateur
      │
      ▼
Supervisor
      │
 ┌────┼─────┬─────┐
 ▼    ▼     ▼     ▼
Plan Res  Write Review
      │
      ▼
Redis Shared State
      │
      ▼
PostgreSQL Memory
```

# Observabilité

In [7]:
import time

start = time.time()

Planner().run()

duration = time.time() - start

print(duration)

7.295608520507812e-05


# Anti-patterns

❌ Chaque agent possède sa mémoire

❌ Sauvegarder toute la conversation

❌ Contexte illimité

❌ Pas de TTL

# Mini Projet

Construire :

- Planner
- Researcher
- Writer

partageant Redis.

# Exercice 1

Créer un shared_state Redis.

# Exercice 2

Ajouter un Supervisor.

# Exercice 3

Sauvegarder les préférences utilisateur.

# Exercice 4

Créer une mémoire PostgreSQL.

# Exercice 5

Dessiner l'architecture complète :
```
OpenAI
↓
Supervisor
↓
Agents
↓
Redis
↓
PostgreSQL
```

# Notes d'architecte

La plupart des développeurs pensent :

"Le problème est le prompt."

En production, le problème est souvent :

- l'état
- la mémoire
- la cohérence
- la coordination

C'est là que se situe une grande partie de la valeur d'un AI Engineer.

# Qu'est-ce que je renvoie au LLM à chaque tour de conversation ?

Beaucoup de développeurs pensent :
```
messages = entire_conversation_history
```
et renvoient tout au LLM.

Cela fonctionne pendant quelques jours.

Puis arrivent :

- coûts élevés
- latence
- contexte saturé
- réponses incohérentes
- hallucinations

## Le vrai problème

Imaginons que ton assistant personnel existe depuis 6 mois.

Tu lui as dit :
```
Je suis développeur Java
Je fais du DevOps
Je veux devenir AI Engineer
J'ai acheté une maison
J'ai un bouledogue français
Je prépare un projet MCP
```
Puis aujourd'hui tu écris :
``
Peux-tu m'aider à concevoir mon serveur MCP ?
``
Faut-il renvoyer au LLM :
```
6 mois de conversations
```
Non.

95% du contexte est inutile.

## Le principe fondamental

À chaque tour :
```
Contexte envoyé
=
Contexte minimal utile
```
et non :
```
Contexte envoyé
=
Tout ce que je sais
```
## Modèle mental

Je recommande de découper le contexte en 4 couches.
```
Prompt système
+
Contexte immédiat
+
Mémoire pertinente
+
Résultats outils
```
## Couche 1 : Prompt système

Stable.

Exemple :
```
Tu es un assistant IA spécialisé
dans l'accompagnement d'un développeur backend
souhaitant devenir AI Engineer.
```
Toujours envoyé.

## Couche 2 : Conversation récente

Par exemple :
```
5 à 20 derniers messages
```
suffisant dans beaucoup de cas.

Exemple :
```
Utilisateur :
Explique MCP

Assistant :
...

Utilisateur :
Comment partager l'état ?
```
Ces échanges sont nécessaires.

## Couche 3 : Mémoire pertinente

C'est ici que la valeur apparaît.

Tu as peut-être 1000 souvenirs.

Mais seulement 5 sont utiles.

Exemple

Question :
```
Comment devenir AI Engineer ?
```
Mémoire récupérée :
```
- développeur Java
- ancien architecte
- compétences DevOps
- formation IA en cours
```
Mémoire ignorée :
```
- bouledogue français
- travaux maison
- tir sportif
```
## Couche 4 : Résultats outils

Exemple :
```
search_document(...)
```
retourne :
```
MCP Specification
version 2026
...
```
Ce résultat doit être injecté.

## Architecture typique
```
Utilisateur
      │
      ▼
Agent
      │
      ▼
Memory Retrieval
      │
      ▼
Construction du contexte
      │
      ▼
LLM
```
## Exemple concret

Utilisateur :
```
Comment construire un MCP Server ?
```
On récupère :

### Historique récent
```
3 derniers messages
```
### Mémoire
```
- développeur Java
- étude des agents
- semaine 3 MCP
```
### Ressource
```
Documentation MCP
```
Puis :
```
context = {

    "system": system_prompt,

    "recent_messages": recent_messages,

    "memories": relevant_memories,

    "resources": retrieved_documents
}
```
## Ce qu'il ne faut surtout pas faire
### Erreur n°1

Tout envoyer
```
all_conversations_since_day_one
```
### Erreur n°2

Tout sauvegarder
```
save_every_message()
```
### Erreur n°3

Tout réinjecter
```
inject_all_memories()
```
## Comment filtrer ?

C'est là qu'intervient le Context Engineering.

Pour chaque information :
```
Cette information
augmente-t-elle la qualité de la réponse ?
```
Si non :
```
Ne pas envoyer
```
### Exemple appliqué à ton cas

Si demain tu demandes :
```
Comment construire un agent MCP ?
```
Je conserverais :
```
✓ Développeur backend Java

✓ Ancien architecte

✓ Forte expérience DevOps

✓ Formation IA en cours

✓ Étude des agents

✓ Étude de MCP

✓ Objectif AI Engineer
```
Je n'enverrais probablement pas :
```
✗ Bouledogue français

✗ Travaux maison

✗ Tir sportif
```
car cela n'aide pas à répondre.

## La règle que j'utilise dans les architectures d'agents

Je découpe le contexte en :
```
Niveau 1
Conversation récente
```
```
Niveau 2
Mémoire utilisateur pertinente
```
```
Niveau 3
Documents RAG
```
```
Niveau 4
Résultats outils
```
et j'applique toujours :
```
Le plus petit contexte
qui permet la meilleure réponse.
```
C'est l'une des différences majeures entre un prototype d'agent et un système réellement exploitable en production. Un AI Engineer passe souvent plus de temps à décider quoi envoyer au LLM qu'à écrire l'appel :
```
client.responses.create(...)
```
car la qualité, le coût et la vitesse du système dépendent directement de cette décision.